# Quantitative Strategy Analysis: Part 1 - Data Loading & Cleaning

**Project Portfolio:** Algorithmic Trading Performance Dashboard  
**Objective:** To ingest, clean, and standardize raw MetaTrader 5 (MT5) execution data to prepare it for advanced quantitative feature engineering and performance evaluation.

### Strategy Context
This data represents a proprietary trading strategy executing across two distinct accounts:
1. **ST01 (Standardized Final Positions):** Starting Capital = £500
2. **ST02 (Standardized Execution Events):** Starting Capital = £2,700

**Risk Model:** The strategy employs a strict **Fixed Fractional Risk Model**, risking exactly **5% of rolling account equity** per trade. Since the account base currency is **GBP**, but assets are priced in USD, specialized data cleaning is required to preserve the exact entry, stop loss, and exit prices to back-calculate precise R-Multiples in subsequent steps.

### Notebook Objectives:
1. Load raw trade logs safely.
2. Standardize erratic MT5 column names for programmatic access.
3. Strip string artifacts (spaces, commas) from financial columns and cast to floats.
4. Parse and format datetime columns accurately, sorting chronologically by exit time (crucial for compounding logic).
5. Export the clean, `interim` data for the Feature Engineering phase.

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import warnings
import os

# Ignore benign warnings for cleaner notebook output
warnings.filterwarnings('ignore')

# Set pandas display options for easier dataframe inspection
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Verify working directory (should be run from the /notebooks/ folder)
print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: c:\MyStuff\tradeSimulator\tranalyzer\trade-analyzer\notebooks


## 1. Data Loading
We begin by loading the raw data from the `data/raw/` directory. 
*Note: Ensure your CSV files are named `ST01.csv` and `ST02.csv`.*

In [2]:
# Define file paths (Update these if your folder structure changes)
st01_path = '../data/raw/ST01.csv'
st02_path = '../data/raw/ST02.csv'

# Load the raw datasets
try:
    df_st01 = pd.read_csv(st01_path)
    df_st02 = pd.read_csv(st02_path)
    
    print(f"✅ Successfully loaded ST01: {df_st01.shape[0]} rows, {df_st01.shape[1]} columns")
    print(f"✅ Successfully loaded ST02: {df_st02.shape[0]} rows, {df_st02.shape[1]} columns")
    
except FileNotFoundError as e:
    print(f"❌ Error loading files: {e}. Please check the file paths in the data/raw/ folder.")
    
# Display a sample of the raw data structure
display(df_st01.head(2))
display(df_st02.head(2))

✅ Successfully loaded ST01: 86 rows, 14 columns
✅ Successfully loaded ST02: 125 rows, 14 columns


,Time,Position,Symbol,Type,Volume,Price,S / L,T / P,Time.1,Price.1,Commission,Swap,Profit,Unnamed: 13
0,2025.10.31 15:37:44,225126954,XTIUSD.s,buy,0.01,60.648,60.648,NaN,2025.11.03 10:58:05,60.642,0,0.0,-0.05,NaN
1,2025.10.31 15:37:44,225126948,XTIUSD.s,buy,0.01,60.648,60.648,NaN,2025.11.03 10:58:05,60.642,0,0.0,-0.05,NaN


,Time,Position,Symbol,Type,Volume,Price,S / L,T / P,Time.1,Price.1,Commission,Swap,Profit,Unnamed: 13
0,2025.11.19 11:40:24,227526468,XTIUSD.s,buy,0.83,60.473,60.25,NaN,2025.11.19 12:01:15,60.243,0,0.0,-148.37,NaN
1,2025.11.19 11:40:24,227526474,XTIUSD.s,buy,0.83,60.473,60.25,NaN,2025.11.19 12:01:15,60.243,0,0.0,-148.37,NaN


## 2. Standardizing Column Names
MT5 exports contain notoriously difficult column names with spaces, slashes, and duplicate names (e.g., two columns named `Time` representing Entry and Exit). We will map these to standard Python `snake_case`.

In [3]:
def standardize_mt5_columns(df):
    """
    Renames and standardizes MT5 columns. Handles the duplicate 'Time' and 'Price' columns.
    """
    df = df.copy()
    
    # Strip whitespace from raw columns just in case
    df.columns = df.columns.str.strip()
    
    # Typical MT5 Structure: 
    # Time | Position | Symbol | Type | Volume | Price | S / L | T / P | Time.1 | Price.1 | Commission | Swap | Profit
    
    # Create a mapping dictionary for clarity
    column_mapping = {
        'Time': 'entry_time',
        'Position': 'ticket',
        'Symbol': 'symbol',
        'Type': 'type',
        'Volume': 'volume',
        'Price': 'entry_price',
        'S / L': 'sl',
        'T / P': 'tp',
        'Time.1': 'exit_time',   # Pandas usually appends .1 to duplicate column names
        'Price.1': 'exit_price', # Pandas usually appends .1 to duplicate column names
        'Commission': 'commission',
        'Swap': 'swap',
        'Profit': 'profit'
    }
    
    # If the CSV didn't automatically append .1 to duplicates, we handle it manually by position
    if list(df.columns).count('Time') > 1:
        cols = list(df.columns)
        first_time = cols.index('Time')
        cols[first_time] = 'entry_time'
        second_time = cols.index('Time') # Finds the next one
        cols[second_time] = 'exit_time'
        
        first_price = cols.index('Price')
        cols[first_price] = 'entry_price'
        second_price = cols.index('Price')
        cols[second_price] = 'exit_price'
        df.columns = cols
    
    # Apply mapping to any remaining columns matching the dictionary
    df.rename(columns=column_mapping, inplace=True)
    
    # Catch-all for any remaining columns: lowercase and replace spaces
    df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('/', '_')
    
    return df

df_st01 = standardize_mt5_columns(df_st01)
df_st02 = standardize_mt5_columns(df_st02)

print("Standardized ST01 Columns:", df_st01.columns.tolist())
print("Standardized ST02 Columns:", df_st02.columns.tolist())

Standardized ST01 Columns: ['entry_time', 'ticket', 'symbol', 'type', 'volume', 'entry_price', 'sl', 'tp', 'exit_time', 'exit_price', 'commission', 'swap', 'profit', 'unnamed:_13']
Standardized ST02 Columns: ['entry_time', 'ticket', 'symbol', 'type', 'volume', 'entry_price', 'sl', 'tp', 'exit_time', 'exit_price', 'commission', 'swap', 'profit', 'unnamed:_13']


## 3. Data Type Formatting & String Cleaning
MT5 often exports numbers as strings with spaces for thousands separators (e.g., `1 445.92`). We must strip these artifacts and cast all financial columns to floats. We will also cast our time columns to Pandas Datetime objects.

In [4]:
def clean_financial_data(df):
    """
    Cleans string artifacts from financial columns, converts to floats, and parses datetimes.
    """
    df = df.copy()
    
    # 1. Parse Datetimes
    time_cols = ['entry_time', 'exit_time']
    for col in time_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format='%Y.%m.%d %H:%M:%S', errors='coerce')
            
    # 2. Clean numeric strings and cast to float
    # We apply this to all columns except ticket, symbol, type, and datetimes
    exclude_cols = ['entry_time', 'exit_time', 'ticket', 'symbol', 'type']
    numeric_cols = [col for col in df.columns if col not in exclude_cols]
    
    for col in numeric_cols:
        if df[col].dtype == 'object':
            # Remove spaces (MT5 thousands separator) and commas
            df[col] = df[col].astype(str).str.replace(' ', '', regex=False).str.replace('\xa0', '', regex=False).str.replace(',', '', regex=False)
        # Convert to numeric, turning empty strings to NaN
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    return df

df_st01 = clean_financial_data(df_st01)
df_st02 = clean_financial_data(df_st02)

# Verify Data Types
print("Data Types after cleaning:")
print(df_st01.dtypes)
print(df_st02.dtypes)

Data Types after cleaning:
entry_time     datetime64[ns]
ticket                  int64
symbol                 object
type                   object
volume                float64
entry_price           float64
sl                    float64
tp                    float64
exit_time      datetime64[ns]
exit_price            float64
commission              int64
swap                  float64
profit                float64
unnamed:_13           float64
dtype: object
entry_time     datetime64[ns]
ticket                  int64
symbol                 object
type                   object
volume                float64
entry_price           float64
sl                    float64
tp                    float64
exit_time      datetime64[ns]
exit_price            float64
commission              int64
swap                  float64
profit                float64
unnamed:_13           float64
dtype: object


## 4. Handling Missing Values and Chronological Sorting
In trading data, `NaN` in a Stop Loss (`sl`) or Take Profit (`tp`) column is normal (indicating no hard stop was set, or it was managed manually). However, we must drop rows missing core execution data. 

**Crucial Step:** Because this strategy relies on a rolling 5% risk of compounding equity, we **must** sort the data by `exit_time`. A trade's profit is not added to the account balance until it closes.

In [5]:
# Drop rows that are completely empty or missing critical execution prices
critical_cols = ['symbol', 'volume', 'entry_price', 'exit_price', 'profit']
df_st01.dropna(subset=critical_cols, inplace=True)
df_st02.dropna(subset=critical_cols, inplace=True)

# Sort chronologically by exit time to allow for accurate rolling balance calculation in Notebook 2
df_st01.sort_values(by='exit_time', ascending=True, inplace=True)
df_st02.sort_values(by='exit_time', ascending=True, inplace=True)

# Reset index after sorting
df_st01.reset_index(drop=True, inplace=True)
df_st02.reset_index(drop=True, inplace=True)

print(f"Final Cleaned ST01 Rows: {df_st01.shape[0]}")
print(f"Final Cleaned ST02 Rows: {df_st02.shape[0]}")

Final Cleaned ST01 Rows: 86
Final Cleaned ST02 Rows: 125


## 5. Export Interim Data
The raw MT5 datasets are now structurally sound, chronologically ordered, and programmatically accessible. We will export these to the `data/interim/` folder. 

In **Notebook 2 (Feature Engineering)**, we will inject the initial capital parameters (£500 and £2,700) and mathematically reconstruct the 5% GBP Risk, Asset Names, and R-Multiples.

In [6]:
# Create interim directory if it doesn't exist
os.makedirs('../data/interim', exist_ok=True)

# Save cleaned datasets
df_st01.to_csv('../data/interim/st01_cleaned.csv', index=False)
df_st02.to_csv('../data/interim/st02_cleaned.csv', index=False)

print("✅ Data cleaning complete. Interim files successfully saved to '../data/interim/'")

✅ Data cleaning complete. Interim files successfully saved to '../data/interim/'
